In [ ]:
# -------------------- Cell 1: Setup and Parameters --------------------
# Goal:
# 1. 指定要处理的原始神经录波文件 (.rhd)
# 2. 设置信号处理、峰检测、聚类的超参数
# 3. 后面所有 cell 都会用这些全局参数

import numpy as np
import os
import gc
import umap
import h5py
from scipy.signal import butter, filtfilt, find_peaks
from sklearn.cluster import DBSCAN

# -------------------- Output folder --------------------
# 所有中间结果、训练数据、模型文件都会丢进这个文件夹
Export_FolderName = "ProcessingTest"


# -------------------- Input data --------------------
# 这 6 个是“高质量”的 .rhd 片段，会被拼接在一起当训练数据来源
# 路径可以是相对路径（如 "data_good/...rhd"）也可以是绝对路径
filenames = [
    "data_good/20250907_0708B_C207D_S_185D_250907_141340.rhd",
]


# -------------------- Time range to analyze --------------------
# 只处理录波里这段时间窗口（单位：秒）
# 目的是避免把整晚整天所有数据都吃进来，先截一个固定长度
time_range_1 = 0        # 起始时间，比如 0 秒
time_range_2 = 10000    # 结束时间，比如 10000 秒。如果录波没这么长就会自动截断


# -------------------- Band-pass filter (spike band) --------------------
# 用来把慢电位漂移/噪声和高频肌电等干掉，只留下spike主频段
order = 3           # 巴特沃斯滤波器阶数
low_cutoff = 250    # Hz，下限，一般 250 Hz 左右对应细胞外动作电位起始频率
high_cutoff = 3000  # Hz，上限，>3 kHz 主要是噪声


# -------------------- Spike detection params --------------------
# 这些参数控制“在哪儿算一次spike”
baseline_width = 5            # 阈值 = baseline_width * 噪声RMS（越大=更严格，只留大spike）
peak_gap = 30                 # 两个spike之间至少要隔多少采样点（防止把同一个spike数两次）
min_peak_quantity = 30        # 这个通道如果总spike太少就跳过（视为inactive通道）
peak_amplitude_limit = 150    # μV，超过这个幅度的当成artifact（比如刺激、电击、电源爆冲）


# -------------------- Spike waveform extraction --------------------
# 每个 spike 我们会截一小段波形出来当样本（喂CNN的“图片”）
peak_width = 0.003   # 秒。比如 3 ms 的窗口（±1.5 ms）；典型细胞外 spike 宽度 0.6~1.2 ms


# -------------------- UMAP (dimensionality reduction) --------------------
# 把每个 spike waveform 压到低维空间，方便做无监督聚类（给CNN当伪label）
n_neighbors = 15
min_dist = 0.1
n_components = 2     # 通常 2D/3D embedding 都够用做DBSCAN


# -------------------- DBSCAN clustering --------------------
# 把 spike waveform 的embedding做密度聚类 -> 近似"不同神经元"
eps = 0.5
min_sample = 5       # 至少多少点形成一类。太小会把噪声也当一类


print("Cell 1 完成：全局参数 & 文件列表已加载 ✅")          

Cell 1 完成：全局参数 & 文件列表已加载 ✅


In [48]:
# -------------------- Cell 2: Load, Concatenate, Trim --------------------
# 目标：
# 1. 读取多个 .rhd / .rhs 文件
# 2. 拼接成一个连续的时间序列
# 3. 根据指定时间范围截取子区段
# 4. 得到 t (时间轴, 秒) 和 amplifier_data (通道 x 时间采样点)，用于后续 spike 检测 / CNN 训练

import numpy as np
import gc

# ----------- 工具函数：读取 Intan 原始数据文件 (.rhd / .rhs) -----------

def load_intan_file(fname):
    """
    读取单个 Intan 采集文件，返回:
    t           : 1D 时间戳数组 (秒)
    amp         : 2D 电压矩阵 [n_channels x n_samples]，单位通常是微伏
    lower_bw    : 采集系统的下限带宽 (Hz)
    upper_bw    : 采集系统的上限带宽 (Hz)
    """
    if fname.endswith(".rhd"):
        import importrhdutilities as imp
        result, ok = imp.load_file(fname)
        if not ok:
            raise RuntimeError(f"读取失败: {fname}")

        t = result["t_amplifier"]
        amp = result["amplifier_data"]
        freq = result["frequency_parameters"]
        lower_bw = freq.get("desired_lower_bandwidth", None)
        upper_bw = freq.get("desired_upper_bandwidth", None)

    elif fname.endswith(".rhs"):
        import importrhsutilities as imp
        result, ok = imp.load_file(fname)
        if not ok:
            raise RuntimeError(f"读取失败: {fname}")

        t = result["t"]
        amp = result["amplifier_data"]
        freq = result["frequency_parameters"]
        lower_bw = freq.get("desired_lower_bandwidth", None)
        upper_bw = freq.get("desired_upper_bandwidth", None)

    else:
        raise ValueError("只支持 .rhd 或 .rhs 文件")

    return t, amp, lower_bw, upper_bw


# ----------- 第一步：逐文件读取并缓存到列表 -----------

all_t_raw = []        # 每段文件自己的时间轴（相对起点）
all_amp_raw = []      # 每段文件的信号矩阵
all_bw_low = []       # 带宽信息(下限)
all_bw_high = []      # 带宽信息(上限)

for fname in filenames:
    print(f"[读取] {fname}")
    t_cur, amp_cur, bw_lo, bw_hi = load_intan_file(fname)

    all_t_raw.append(t_cur)
    all_amp_raw.append(amp_cur)
    all_bw_low.append(bw_lo)
    all_bw_high.append(bw_hi)

# 简单 check：假设所有片段都是同一块电极板，通道数必须一致
n_channels_list = [a.shape[0] for a in all_amp_raw]
if len(set(n_channels_list)) != 1:
    raise RuntimeError(f"不同文件的通道数量不一致: {n_channels_list}")
n_chan = n_channels_list[0]


# ----------- 第二步：把多段文件在时间轴上首尾拼接 -----------

# 我们要做人为的“全局时间轴 t_full”，让第二个文件的时间接在第一个文件后面
t_full_list = []
offset = 0.0
for idx, t_seg in enumerate(all_t_raw):
    # 把这个段的时间平移成连续的 (前一段最后时间 + 当前段相对时间0开始)
    t_shifted = t_seg - t_seg[0] + offset
    t_full_list.append(t_shifted)

    # 下一个 offset = 当前这个段的最后一个时间
    offset = t_shifted[-1]

# 拼接成一个长时间轴
t_full = np.concatenate(t_full_list, axis=0)

# 同时把对应的电压信号矩阵按列拼接
# 每段 amp_raw 的形状是 [n_chan x n_samples_segment]
amp_full = np.concatenate(all_amp_raw, axis=1)  # 按时间方向拼接（列拼）

# 清理中间变量以省内存
del all_t_raw, all_amp_raw, t_full_list
gc.collect()

# ----------- 第三步：计算采样率等基本信息 -----------

total_time_full = t_full[-1] - t_full[0]
n_time_full = amp_full.shape[1]
sampling_rate = int(round(n_time_full / total_time_full))

print("\n===== 数据基本信息 =====")
print(f"通道数 (n_chan): {n_chan}")
print(f"总采样点数 (n_time): {n_time_full}")
print(f"总时长 (s): {total_time_full:.3f}")
print(f"采样率 (Hz): {sampling_rate}")
print(f"系统带宽(第1段): {all_bw_low[0]} ~ {all_bw_high[0]} Hz")
print("========================\n")


# ----------- 第四步：按照设定的时间范围裁剪 -----------

# 注意：time_range_2 可能比真实记录时间长，所以要截断
t_start = max(0, time_range_1)
t_end   = min(time_range_2, total_time_full)

# 把秒 -> 索引
idx_start = int(t_start * sampling_rate)
idx_end   = int(t_end   * sampling_rate)

t = t_full[idx_start:idx_end] - t_full[idx_start]  # 重新从0开始计时
amplifier_data = amp_full[:, idx_start:idx_end]    # 对每个通道截同一时间段

total_time = t[-1]  # 现在实际截取的时长
total_channel = amplifier_data.shape[0]
total_data_point = amplifier_data.shape[1]

print("===== 裁剪后用于后续处理的数据块 =====")
print(f"裁剪区间: {t_start:.3f}s ~ {t_end:.3f}s")
print(f"实际时长: {total_time:.3f} s")
print(f"保留通道数: {total_channel}")
print(f"保留采样点数/通道: {total_data_point}")
print("====================================\n")

# ----------- 第五步：把关键变量留在内存中，供下一步使用 -----------

# 后面 Cell 3 (spike检测+聚类) 会直接使用这些全局变量：
# - t                (1D 时间轴, 单位秒)
# - amplifier_data   (2D 信号矩阵: 通道 x 时间点)
# - total_time, total_channel, total_data_point, sampling_rate
# - Export_FolderName
# - 以及上面 Cell 1 中定义的参数(滤波、阈值、UMAP等)

print("Cell 2 完成：数据已在内存中准备好，可进入 spike 检测。")

[读取] data_good/20250907_0708B_C207D_S_185D_250907_141340.rhd

Reading Intan Technologies RHD Data File, Version 3.4

Found 240 amplifier channels.
Found 15 auxiliary input channels.
Found 0 supply voltage channels.
Found 0 board ADC channels.
Found 0 board digital input channels.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

File contains 60.000 seconds of data.  Amplifiers were sampled at 20.00 kS/s.

Allocating memory for data...
Reading data from file...
10% done...
20% done...
30% done...
40% done...
50% done...
60% done...
70% done...
80% done...
90% done...
Parsing data...
No missing timestamps in data.
Done!  Elapsed time: 6.4 seconds

===== 数据基本信息 =====
通道数 (n_chan): 240
总采样点数 (n_time): 1200000
总时长 (s): 60.000
采样率 (Hz): 20000
系统带宽(第1段): 0.10000000149011612 ~ 3000.0 Hz

===== 裁剪后用于后续处理的数据块 =====
裁剪区间: 0.000s ~ 60.000s
实际时长: 60.000 s
保留通道数: 240
保留采样点数/通道: 1199999

Cell 2 完成：数据已在内存中准备好，可进入 spike 检测。


In [49]:
# -------------------- Cell 3: Filtering, Spike Detection, Clustering --------------------
# 主要功能：
# 1️⃣ 对每个通道进行带通滤波（去除低频伪迹和高频噪声）
# 2️⃣ 基于阈值检测并提取 spike 波形段（spike waveform segments）
# 3️⃣ 使用 UMAP + DBSCAN 对 spike 进行无监督聚类（初步 spike sorting）
# 4️⃣ 输出每个通道的 spike 片段和对应的聚类标签，为后续 CNN 训练做准备

from scipy.signal import butter, filtfilt

# -------------------- 定义辅助函数 --------------------

def make_bandpass(sampling_rate, low, high, order=3):
    """
    构造 Butterworth 带通滤波器，用于提取 spike 频段信号（典型范围 250–3000 Hz）。
    输入：
        sampling_rate : 采样率（Hz）
        low, high      : 下、上截止频率（Hz）
        order          : 滤波器阶数
    输出：
        b, a : 滤波器系数
    """
    nyquist = 0.5 * sampling_rate
    b, a = butter(order, [low / nyquist, high / nyquist], btype='band')
    return b, a


def estimate_rms(data, b, a):
    """
    对每个通道的信号计算 RMS 噪声幅度，用作动态阈值基准。
    输入：
        data : shape=(n_channels, n_timepoints) 的原始波形数据
        b, a : 滤波器系数
    输出：
        rms  : 每个通道对应的 RMS 噪声值数组
    """
    n_chan = data.shape[0]
    rms = np.zeros(n_chan)
    for i in range(n_chan):
        filt = filtfilt(b, a, data[i])
        rms[i] = np.std(filt)
    return rms


def extract_spikes(filtered, thr, peak_gap, limit, peak_width, rate, t):
    """
    从滤波后的单通道信号中提取 spike：
    - 阈值检测正负峰
    - 去掉边缘区段
    - 截取固定宽度的 spike 片段
    输入：
        filtered   : 当前通道的滤波信号
        thr        : 检测阈值（通常为 baseline_width × RMS）
        peak_gap   : 最小峰间距（样点数）
        limit      : 峰值上限（µV），防止伪迹
        peak_width : 单个 spike 片段总时长（秒）
        rate       : 采样率
        t          : 对应的时间轴
    输出：
        segs  : shape=(n_spikes, window_len) 的 spike 片段矩阵
        times : 对应的 spike 发生时间（秒）
    """
    # 检测正、负峰
    pos, _ = find_peaks(filtered, height=(thr, limit), distance=peak_gap)
    neg, _ = find_peaks(-filtered, height=(thr, limit), distance=peak_gap)
    idx = np.concatenate([pos, neg])
    idx.sort()

    # 截取每个 spike 波形窗口
    half = int(peak_width / 2 * rate)
    valid = [x for x in idx if half <= x <= len(filtered) - half]
    if len(valid) < min_peak_quantity:
        # 若该通道spike过少，返回空阵列
        return np.zeros((0, 2 * half)), []

    segs = np.zeros((len(valid), 2 * half))
    for j, c in enumerate(valid):
        segs[j] = filtered[c - half:c + half]
    return segs, t[valid]


# -------------------- Step 1. 提取目标时间段信号 --------------------

# 将时间区间 (time_range_1, time_range_2) 转换为采样点索引
start = int(time_range_1 * sampling_rate)
end = int(min(total_time, time_range_2) * sampling_rate)

t_seg = t_full[start:end] - t_full[start]   # 时间归一到 0 起点
amp_seg = amp_full[:, start:end]            # 截取对应数据段

# -------------------- Step 2. 设计滤波器 & 估算噪声 --------------------

b, a = make_bandpass(sampling_rate, low_cutoff, high_cutoff, order)
rms_noise = estimate_rms(amp_seg, b, a)  # 每通道 RMS 噪声估计

# -------------------- Step 3. 遍历通道进行 spike 检测与聚类 --------------------

X_list, y_list = [], []   # 用于存放所有 spike 波形与聚类标签

for ch in range(n_chan):
    # 带通滤波
    filt = filtfilt(b, a, amp_seg[ch])

    # 阈值检测 + 波形截取
    segs, times = extract_spikes(
        filt, baseline_width * rms_noise[ch], peak_gap,
        peak_amplitude_limit, peak_width, sampling_rate, t_seg
    )
    if segs.shape[0] == 0:
        continue  # 若该通道无有效 spike，则跳过

    # --- UMAP 降维 ---
    # 将高维 spike 波形映射至低维空间以便聚类（常见 n_components=2）
    reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=n_components)
    emb = reducer.fit_transform(segs)

    # --- DBSCAN 聚类 ---
    # 基于密度的无监督聚类算法，可自动识别离群点 (-1)
    db = DBSCAN(eps=eps, min_samples=min_sample)
    labels = db.fit_predict(emb)

    # 保存结果
    X_list.append(segs)
    y_list.append(labels)

    print(f"Channel {ch:03d}: 检测到 {len(labels)} 个spike，聚类数={len(np.unique(labels))}")

print("\n✅ Spike detection + clustering 完成。\n")

Channel 001: 检测到 47 个spike，聚类数=3
Channel 002: 检测到 698 个spike，聚类数=2
Channel 003: 检测到 480 个spike，聚类数=2
Channel 004: 检测到 143 个spike，聚类数=2
Channel 005: 检测到 61 个spike，聚类数=3
Channel 006: 检测到 46 个spike，聚类数=4
Channel 008: 检测到 170 个spike，聚类数=2
Channel 010: 检测到 152 个spike，聚类数=2
Channel 011: 检测到 35 个spike，聚类数=1
Channel 018: 检测到 169 个spike，聚类数=1
Channel 019: 检测到 78 个spike，聚类数=2
Channel 020: 检测到 50 个spike，聚类数=3
Channel 022: 检测到 66 个spike，聚类数=3
Channel 023: 检测到 137 个spike，聚类数=2
Channel 024: 检测到 191 个spike，聚类数=2
Channel 026: 检测到 235 个spike，聚类数=1
Channel 028: 检测到 186 个spike，聚类数=2
Channel 029: 检测到 182 个spike，聚类数=2
Channel 034: 检测到 172 个spike，聚类数=1
Channel 035: 检测到 66 个spike，聚类数=5
Channel 037: 检测到 151 个spike，聚类数=4
Channel 038: 检测到 68 个spike，聚类数=8
Channel 044: 检测到 104 个spike，聚类数=2
Channel 045: 检测到 93 个spike，聚类数=2
Channel 046: 检测到 95 个spike，聚类数=2
Channel 047: 检测到 86 个spike，聚类数=3
Channel 052: 检测到 161 个spike，聚类数=5
Channel 058: 检测到 209 个spike，聚类数=2
Channel 061: 检测到 123 个spike，聚类数=3
Channel 065: 检测到 655 个spik

In [50]:
# -------------------- Cell 4: Save CNN Training Dataset --------------------
# 主要功能：
# 1️⃣ 将前面 Cell 3 中各通道的 spike 波形与聚类标签合并为统一训练集
# 2️⃣ 保存为压缩的 .npz 文件 (便于后续 CNN 模型训练加载)
# ---------------------------------------------------------------------------

# ---- Step 1. 合并所有通道的 spike 波形与标签 ----
# X_list[i] : 第 i 个通道的 spike 波形矩阵，shape ≈ [n_spikes_i, window_len]
# y_list[i] : 第 i 个通道的对应聚类标签，shape ≈ [n_spikes_i]
# 通过 np.concatenate 拼接成统一数组：
#   X.shape ≈ [Σ n_spikes_i, window_len]
#   y.shape ≈ [Σ n_spikes_i]
X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

# ---- Step 2. 创建导出文件夹并保存 ----
os.makedirs(Export_FolderName, exist_ok=True)
save_path = os.path.join(Export_FolderName, "CNN_training_data.npz")

# 保存为压缩格式，既节省空间又方便后续读取：
# 读取时可使用：
#     data = np.load("CNN_training_data.npz")
#     X, y = data["X"], data["y"]
np.savez_compressed(save_path, X=X, y=y)

# ---- Step 3. 输出文件信息 ----
print(f"✅ CNN 训练数据已保存至: {save_path}")
print(f"数据维度: X = {X.shape}, y = {y.shape}")

✅ CNN 训练数据已保存至: ProcessingTest/CNN_training_data.npz
数据维度: X = (23094, 60), y = (23094,)


In [51]:
# -------------------- Cell 5: CNN Spike Classifier Training / Fine-tune --------------------
# 目的：
# (1) 从 Cell 4 导出的 CNN_training_data.npz 读取 spike 波形 (X) 和聚类标签 (y)
# (2) 去掉噪声类（DBSCAN 的 label = -1）
# (3) 训练 / 或微调 一个轻量级1D CNN分类器，用于将 spike 波形归类到不同神经元单元
# (4) 将模型权重、类别映射关系(label_remap) 持久化到 ProcessingTest/cnn_spike_classifier.pth
#
# 支持增量学习：
# - 如果已存在旧模型：
#       * 读取旧的 feature extractor 卷积层权重
#       * 保持旧的 label_remap（cluster -> class_id）
#       * 给新出现的 cluster 分配新的 class_id
#       * 新建一个更大的分类头
# - 如果是第一次训练：
#       * 全部随机初始化
#
# 每次你跑 Cell1~4 得到一批 (X,y) 新数据后，再跑这个 Cell5，就会把新数据“并入”模型。

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ---------------- Step 0. 读取这一批要训练的数据 ----------------
data_path = os.path.join(Export_FolderName, "CNN_training_data.npz")
model_path = os.path.join(Export_FolderName, "cnn_spike_classifier.pth")

loaded = np.load(data_path)
X = loaded["X"]   # shape: [num_spikes, window_len]  spike波形片段
y = loaded["y"]   # shape: [num_spikes]              DBSCAN聚类标签，可能包含-1

print("原始拼接后: ", X.shape, y.shape)

# 去掉 DBSCAN 标的噪声点（cluster = -1 代表离群/伪峰）
valid_mask = (y != -1)
X = X[valid_mask]
y = y[valid_mask]

if X.shape[0] == 0:
    raise RuntimeError("这一批数据全是噪声(-1)，没有可训练样本，无法更新模型。")

print("去掉噪声后:", X.shape[0], "spikes 可用于监督训练")

# 这批新数据里有哪些 cluster id
unique_new_clusters = np.unique(y)

# ---------------- Step 1. 准备旧模型的元信息（如果有的话） ----------------
has_old_model = os.path.exists(model_path)

if has_old_model:
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    old_state_dict = checkpoint["model_state_dict"]   # 旧模型权重
    old_label_remap = checkpoint["label_remap"]       # dict: {cluster_id:int -> class_idx:int}
    old_num_classes = checkpoint["num_classes"]       # 上次训练后有多少个类
    print(f"检测到已有模型，旧类别数 = {old_num_classes}")
else:
    checkpoint = None
    old_state_dict = None
    old_label_remap = {}
    old_num_classes = 0
    print("未发现旧模型，将创建新模型")

# ---------------- Step 2. 合并/更新 label_remap ----------------
# 我们维持一个 "cluster_id -> class_index" 的全局映射。
# 已出现过的cluster沿用原class id；新出现的cluster分配新class id。

label_remap = dict(old_label_remap)  # 复制旧映射，不破坏历史
next_class_idx = old_num_classes     # 新class从这个编号往后排

for c in unique_new_clusters:
    c_int = int(c)
    if c_int not in label_remap:
        label_remap[c_int] = next_class_idx
        next_class_idx += 1

num_classes = next_class_idx
print("合并后类别总数 =", num_classes)
print("label_remap =", label_remap)

# 把 y（cluster id）映射成连续的 class index [0 .. num_classes-1]
y_mapped = np.array([label_remap[int(v)] for v in y], dtype=np.int64)

# ---------------- Step 3. train/test 划分 ----------------
num_samples = X.shape[0]
idx = np.random.permutation(num_samples)

# 如果样本很少，比如 <10，就尽量留至少1个样本做“测试”
if num_samples >= 10:
    split = int(num_samples * 0.8)
else:
    split = max(num_samples - 1, 1)

train_idx = idx[:split]
test_idx  = idx[split:]

X_train = X[train_idx]
y_train = y_mapped[train_idx]
X_test  = X[test_idx]
y_test  = y_mapped[test_idx]

# 转成 PyTorch Tensor 并加 channel 维度
# Conv1d 期望输入 [batch, channels, length]，我们只有1个通道
X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32).unsqueeze(1)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

input_len = X.shape[1]  # spike窗口长度（例如 60 点）

# ---------------- Step 4. 定义模型结构 ----------------
class SpikeCNN(nn.Module):
    """
    轻量级 1D CNN，用于尖峰波形分类。
    - feature: 卷积特征提取器，提取波形形状的局部/全局模式
    - classifier: 线性分类头，把 64维特征 -> num_classes
    说明：
    - 我们使用 AdaptiveAvgPool1d(1)，所以不用关心输入波形长度变化
    - 这个结构非常适合边缘推理/实时检测
    """
    def __init__(self, n_classes):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),  # 时间长度 /2

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),  # 再 /2

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),  # -> [B,64,1]
        )
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.feature(x)      # [B,64,1]
        x = x.squeeze(-1)        # [B,64]
        x = self.classifier(x)   # [B,num_classes]
        return x

# 建一个“当前最新类别数”的模型
model = SpikeCNN(num_classes)

# ---------------- Step 5. 如果存在旧模型 -> 只加载特征提取层权重 ----------------
if has_old_model:
    # 我们不直接 load_state_dict(old_state_dict) 因为 classifier 尺寸变了
    # 我们手动挑出 feature.* 的权重，跳过 classifier.*
    filtered_state = {
        k: v for k, v in old_state_dict.items()
        if k.startswith("feature.")
    }

    # 丢进 model。strict=False 允许我们只加载一部分 keys。
    missing_keys, unexpected_keys = model.load_state_dict(filtered_state, strict=False)
    print("已加载旧特征层(卷积)权重。")
    print("missing_keys (通常是 classifier.*):", missing_keys)
    print("unexpected_keys:", unexpected_keys)
else:
    print("无旧模型，使用随机初始化的特征提取器和分类头。")

# ---------------- Step 6. 训练设置 ----------------
criterion = nn.CrossEntropyLoss()

# 增量学习阶段我们用较小的学习率，避免把旧特征一下子全洗掉
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def evaluate_accuracy(net, dataloader):
    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in dataloader:
            logits = net(xb)
            pred = torch.argmax(logits, dim=1)
            correct += (pred == yb).sum().item()
            total += yb.numel()
    return (correct / total) if total > 0 else 0.0

# ---------------- Step 7. 训练若干 epoch ----------------
num_epochs = 5  # 先短训几轮，主要是把新类“挂到”模型里
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    test_acc = evaluate_accuracy(model, test_loader)

    print(f"epoch {epoch}: loss={avg_loss:.4f}, test_acc={test_acc:.3f}")

# ---------------- Step 8. 保存最新模型 ----------------
torch.save({
    "model_state_dict": model.state_dict(),
    "num_classes": num_classes,
    "label_remap": label_remap,
}, model_path)

print(f"\n模型已保存到: {model_path}")
print("当前全局类别总数 =", num_classes)
print("label_remap =", label_remap)

原始拼接后:  (23094, 60) (23094,)
去掉噪声后: 22734 spikes 可用于监督训练
检测到已有模型，旧类别数 = 6
合并后类别总数 = 7
label_remap = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6}
已加载旧特征层(卷积)权重。
missing_keys (通常是 classifier.*): ['classifier.weight', 'classifier.bias']
unexpected_keys: []
epoch 0: loss=1.5209, test_acc=0.682
epoch 1: loss=0.6727, test_acc=0.684
epoch 2: loss=0.6318, test_acc=0.754
epoch 3: loss=0.6072, test_acc=0.754
epoch 4: loss=0.5916, test_acc=0.753

模型已保存到: ProcessingTest/cnn_spike_classifier.pth
当前全局类别总数 = 7
label_remap = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6}
